# PDF OCR Extractor - Page 4 Detailed Analysis

This notebook extracts detailed information from **page 4** of a PDF file including:
- Text content via OCR
- Checkbox detection and counting
- Word count statistics
- Downloadable .txt and .docx files

**Features:**
- OpenCV image preprocessing for optimal OCR
- Advanced checkbox detection algorithm
- Console output with complete summary

## Step 1: Install Required Libraries

In [ ]:
# Install all required packages
!pip install -q pdf2image pillow opencv-python pytesseract python-docx
!apt-get install -y poppler-utils tesseract-ocr

print("✓ All libraries installed successfully!")

## Step 2: Import Libraries

In [ ]:
import cv2
import numpy as np
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Pt, RGBColor
from datetime import datetime
import io
from google.colab import files
import os

print("✓ All libraries imported successfully!")

## Step 3: Upload PDF File

In [ ]:
print("📤 Please upload your PDF file...")
uploaded = files.upload()

# Get the uploaded filename
pdf_filename = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {pdf_filename}")

## Step 4: Convert PDF Page 4 to Image

In [ ]:
print("\n📄 Converting PDF page 4 to image...")

# Convert only page 4 (first_page=4, last_page=4)
# Using high DPI for better OCR quality
pages = convert_from_path(pdf_filename, dpi=300, first_page=4, last_page=4)

if len(pages) == 0:
    print("❌ Error: Page 4 not found in PDF!")
else:
    page_4_image = pages[0]
    
    # Convert PIL Image to numpy array for OpenCV
    image_np = np.array(page_4_image)
    
    # Convert RGB to BGR (OpenCV format)
    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    
    print(f"✓ Page 4 extracted successfully!")
    print(f"  Image size: {image_bgr.shape[1]} x {image_bgr.shape[0]} pixels")
    
    # Display original image
    plt.figure(figsize=(12, 16))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title("Original Page 4", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Step 5: Preprocess Image with OpenCV (Retry)

In [ ]:
print("\n🔧 Preprocessing image with OpenCV...\n")

# Step 1: Convert to grayscale
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
print("  ✓ Converted to grayscale")

# Step 2: Apply bilateral filter to reduce noise while keeping edges sharp
denoised = cv2.bilateralFilter(gray, 9, 75, 75)
print("  ✓ Applied bilateral filtering for noise reduction")

# Step 3: Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(denoised)
print("  ✓ Enhanced contrast using CLAHE")

# Step 4: Apply adaptive thresholding
thresh = cv2.adaptiveThreshold(
    enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)
print("  ✓ Applied adaptive thresholding")

# Step 5: Morphological operations to clean up
kernel = np.ones((1, 1), np.uint8)
processed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
print("  ✓ Applied morphological operations")

print("\n✓ Preprocessing complete!\n")

# Display preprocessing steps
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('OpenCV Preprocessing Pipeline', fontsize=18, fontweight='bold')

images_to_show = [
    (cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), 'Original'),
    (gray, 'Grayscale'),
    (denoised, 'Denoised'),
    (enhanced, 'Contrast Enhanced'),
    (thresh, 'Adaptive Threshold'),
    (processed, 'Final Processed')
]

for idx, (img, title) in enumerate(images_to_show):
    row = idx // 3
    col = idx % 3
    axes[row, col].imshow(img, cmap='gray' if idx > 0 else None)
    axes[row, col].set_title(title, fontsize=14, fontweight='bold')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## Step 6: Detect and Count Checkboxes

In [ ]:
print("\n☑️  Detecting checkboxes...\n")

# Create a copy for visualization
checkbox_image = image_bgr.copy()

# Find contours
contours, _ = cv2.findContours(processed, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

checkbox_count = 0
checkbox_locations = []

# Filter contours to find checkboxes
for contour in contours:
    # Get bounding rectangle
    x, y, w, h = cv2.boundingRect(contour)
    
    # Filter based on size and aspect ratio
    # Typical checkbox: 10-50 pixels, aspect ratio close to 1:1
    aspect_ratio = float(w) / h if h > 0 else 0
    area = w * h
    
    # Checkbox criteria: square-ish shape, reasonable size
    if (10 < w < 80 and 10 < h < 80 and 
        0.7 < aspect_ratio < 1.3 and 
        100 < area < 6400):
        
        # Additional check: perimeter to area ratio (squares have specific ratio)
        perimeter = cv2.arcLength(contour, True)
        if area > 0:
            circularity = (4 * np.pi * area) / (perimeter * perimeter) if perimeter > 0 else 0
            
            # Squares have circularity around 0.785
            if circularity > 0.6:
                checkbox_count += 1
                checkbox_locations.append((x, y, w, h))
                
                # Draw rectangle around detected checkbox
                cv2.rectangle(checkbox_image, (x, y), (x + w, y + h), (0, 255, 0), 3)
                
                # Add checkbox number
                cv2.putText(checkbox_image, str(checkbox_count), (x - 5, y - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

print(f"✓ Total checkboxes detected: {checkbox_count}")
print(f"  Checkbox locations: {len(checkbox_locations)} found\n")

# Display image with detected checkboxes
plt.figure(figsize=(12, 16))
plt.imshow(cv2.cvtColor(checkbox_image, cv2.COLOR_BGR2RGB))
plt.title(f"Detected Checkboxes: {checkbox_count}", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# Print checkbox coordinates
if checkbox_count > 0:
    print("\nCheckbox Details:")
    for i, (x, y, w, h) in enumerate(checkbox_locations, 1):
        print(f"  Checkbox {i}: Position ({x}, {y}), Size {w}x{h}")

## Step 7: Perform OCR and Extract Text

In [ ]:
print("\n📝 Performing OCR text extraction...\n")

# Configure Tesseract for best results
custom_config = r'--oem 3 --psm 6'

# Perform OCR on the processed image
extracted_text = pytesseract.image_to_string(processed, config=custom_config)

# Also get detailed data with confidence scores
ocr_data = pytesseract.image_to_data(processed, output_type=pytesseract.Output.DICT, config=custom_config)

# Calculate average confidence
confidences = [int(conf) for conf in ocr_data['conf'] if conf != '-1']
avg_confidence = sum(confidences) / len(confidences) if confidences else 0

print(f"✓ OCR extraction complete!")
print(f"  Average confidence: {avg_confidence:.2f}%")
print(f"  Text length: {len(extracted_text)} characters\n")

## Step 8: Calculate Word Count

In [ ]:
print("\n🔢 Calculating word count...\n")

# Count words (split by whitespace and filter empty strings)
words = extracted_text.split()
word_count = len(words)

# Count lines
lines = extracted_text.split('\n')
line_count = len([line for line in lines if line.strip()])

# Count characters (excluding whitespace)
char_count = len(extracted_text.replace(' ', '').replace('\n', ''))

print(f"✓ Statistics calculated:")
print(f"  Total words: {word_count}")
print(f"  Total lines: {line_count}")
print(f"  Total characters (no spaces): {char_count}")
print(f"  Total checkboxes: {checkbox_count}")

## Step 9: Generate Output and Print Summary

In [ ]:
print("\n" + "=" * 80)
print(" " * 20 + "PDF OCR EXTRACTION SUMMARY")
print("=" * 80)
print(f"\nPDF File: {pdf_filename}")
print(f"Page Analyzed: 4")
print(f"Extraction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "-" * 80)
print("STATISTICS")
print("-" * 80)
print(f"Total Words:      {word_count:,}")
print(f"Total Lines:      {line_count:,}")
print(f"Total Characters: {char_count:,}")
print(f"Total Checkboxes: {checkbox_count:,}")
print(f"OCR Confidence:   {avg_confidence:.2f}%")
print("\n" + "-" * 80)
print("EXTRACTED TEXT")
print("-" * 80)
print(extracted_text)
print("\n" + "=" * 80)
print(" " * 25 + "END OF SUMMARY")
print("=" * 80 + "\n")

## Step 10: Generate and Download Text File (.txt)

In [ ]:
print("\n📄 Generating text file...\n")

# Create text file content
txt_filename = f"page4_extraction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

txt_content = f"""PDF OCR EXTRACTION REPORT
{'=' * 80}

Source PDF: {pdf_filename}
Page Number: 4
Extraction Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

{'=' * 80}
STATISTICS
{'=' * 80}

Total Words:      {word_count:,}
Total Lines:      {line_count:,}
Total Characters: {char_count:,}
Total Checkboxes: {checkbox_count:,}
OCR Confidence:   {avg_confidence:.2f}%

{'=' * 80}
EXTRACTED TEXT
{'=' * 80}

{extracted_text}

{'=' * 80}
END OF REPORT
{'=' * 80}
"""

# Write to file
with open(txt_filename, 'w', encoding='utf-8') as f:
    f.write(txt_content)

print(f"✓ Text file created: {txt_filename}")
print(f"  File size: {os.path.getsize(txt_filename)} bytes")

# Download the file
print("\n⬇️  Downloading text file...")
files.download(txt_filename)
print("✓ Download started!")

## Step 11: Generate and Download Word File (.docx)

In [ ]:
print("\n📘 Generating Word document...\n")

# Create Word document
docx_filename = f"page4_extraction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
doc = Document()

# Add title
title = doc.add_heading('PDF OCR Extraction Report', 0)
title.alignment = 1  # Center alignment

# Add metadata section
doc.add_heading('Document Information', 1)
info_table = doc.add_table(rows=3, cols=2)
info_table.style = 'Light Grid Accent 1'

info_data = [
    ('Source PDF:', pdf_filename),
    ('Page Number:', '4'),
    ('Extraction Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
]

for idx, (key, value) in enumerate(info_data):
    row = info_table.rows[idx]
    row.cells[0].text = key
    row.cells[1].text = value

doc.add_paragraph()  # Spacing

# Add statistics section
doc.add_heading('Extraction Statistics', 1)
stats_table = doc.add_table(rows=5, cols=2)
stats_table.style = 'Light Grid Accent 1'

stats_data = [
    ('Total Words:', f"{word_count:,}"),
    ('Total Lines:', f"{line_count:,}"),
    ('Total Characters:', f"{char_count:,}"),
    ('Total Checkboxes:', f"{checkbox_count:,}"),
    ('OCR Confidence:', f"{avg_confidence:.2f}%")
]

for idx, (key, value) in enumerate(stats_data):
    row = stats_table.rows[idx]
    row.cells[0].text = key
    row.cells[1].text = value

doc.add_paragraph()  # Spacing

# Add extracted text section
doc.add_heading('Extracted Text', 1)
text_para = doc.add_paragraph(extracted_text)
text_para.style = 'Normal'

# Add footer
doc.add_paragraph()  # Spacing
footer = doc.add_paragraph('\n' + '=' * 80)
footer.add_run('\nGenerated by PDF OCR Extractor').bold = True
footer.alignment = 1  # Center alignment

# Save document
doc.save(docx_filename)

print(f"✓ Word document created: {docx_filename}")
print(f"  File size: {os.path.getsize(docx_filename)} bytes")

# Download the file
print("\n⬇️  Downloading Word document...")
files.download(docx_filename)
print("✓ Download started!")

## Final Task Summary

In [ ]:
print("\n" + "*" * 80)
print(" " * 25 + "✅ ALL TASKS COMPLETED!")
print("*" * 80)
print("\nCompleted Tasks:")
print("  ✓ PDF uploaded and page 4 extracted")
print("  ✓ Image preprocessing with OpenCV")
print("  ✓ Checkbox detection and counting")
print("  ✓ OCR text extraction")
print("  ✓ Word count and statistics calculated")
print("  ✓ Text file (.txt) generated and downloaded")
print("  ✓ Word file (.docx) generated and downloaded")
print("  ✓ Complete summary printed to console")
print("\nOutput Files:")
print(f"  📄 {txt_filename}")
print(f"  📘 {docx_filename}")
print("\nKey Results:")
print(f"  📊 Words: {word_count:,} | Checkboxes: {checkbox_count:,} | Confidence: {avg_confidence:.2f}%")
print("\n" + "*" * 80 + "\n")